# Perbandingan Model SC-WEAT Bahasa Indonesia — test_v3

Notebook ini membandingkan asosiasi intrinsik gender–jabatan pada dua model embedding statis Bahasa Indonesia. `cc.id.300.bin` dipertahankan sebagai baseline dan `wiki.id.bin` digunakan sebagai model pembanding. Data Jabatan, lexical set gender, formula SC-WEAT single-target, permutation test, dan aturan preprocessing dibuat sama pada kedua model.

Perbandingan ini menguji sensitivitas hasil terhadap sumber korpus dan konfigurasi pelatihan, bukan mengisolasi efek arsitektur secara murni. Hasil yang signifikan tidak ditargetkan melalui pemilihan model setelah melihat p-value. Model yang dipakai ditetapkan sebelum analisis dan seluruh hasil model dilaporkan.

## 1. Import, sumber data, dan konfigurasi

Model biner tidak disimpan di Git karena ukurannya besar. Letakkan kedua model di folder `models/` sebelum menjalankan notebook:

- `cc.id.300.bin`: pretrained fastText dari Common Crawl dan Wikipedia, sumber resmi `https://fasttext.cc/docs/en/crawl-vectors.html`.
- `wiki.id.bin`: pretrained fastText dari Wikipedia, sumber resmi `https://fasttext.cc/docs/en/pretrained-vectors.html`.

Kedua model sama-sama embedding statis berdimensi 300, tetapi berbeda pada sumber korpus dan objective pelatihan. Perbedaan itu harus disebutkan ketika hasil ditafsirkan.

In [ ]:
import hashlib
import importlib.metadata
import itertools
import re
import sys
from pathlib import Path

import fasttext
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown
from scipy.stats import kendalltau, spearmanr, wilcoxon
from statsmodels.stats.multitest import multipletests

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data"
MODEL_DIR = REPO_ROOT / "models"
OUTPUT_DIR = REPO_ROOT / "outputs"

ANALYSIS_DATA = DATA_DIR / "test_v3_kbji_profesional_analysis.csv"
GENDER_DATA = DATA_DIR / "test_v3_istilah_gender.csv"

MODEL_SPECS = pd.DataFrame([
    {
        "model_id": "cc_crawl",
        "model_name": "cc.id.300.bin",
        "path": MODEL_DIR / "cc.id.300.bin",
        "corpus": "Common Crawl + Wikipedia",
        "training_note": "CBOW dengan position weights",
        "source_url": "https://fasttext.cc/docs/en/crawl-vectors.html",
    },
    {
        "model_id": "wiki",
        "model_name": "wiki.id.bin",
        "path": MODEL_DIR / "wiki.id.bin",
        "corpus": "Wikipedia",
        "training_note": "Skip-gram",
        "source_url": "https://fasttext.cc/docs/en/pretrained-vectors.html",
    },
])

display(MODEL_SPECS.assign(path=MODEL_SPECS.path.astype(str)))
missing_models = MODEL_SPECS.loc[~MODEL_SPECS.path.map(Path.exists)]
if not missing_models.empty:
    display(missing_models[["model_id", "model_name", "path", "source_url"]].assign(path=missing_models.path.astype(str)))
    raise FileNotFoundError("Letakkan seluruh model v3 di folder models/ sebelum melanjutkan.")

## 2. Membaca data KBJI dan lexical set

Notebook membaca hasil notebook pemrosesan v3. Analisis utama tetap berada pada Jabatan enam digit di Subgolongan Pokok 21–26. CSV lexical set menjadi satu-satunya sumber pasangan istilah gender; notebook tidak mendefinisikan ulang pasangan tersebut.

In [ ]:
df_analysis = pd.read_csv(
    ANALYSIS_DATA,
    dtype={
        "occupation_code": str,
        "major_group_code": str,
        "submajor_group_code": str,
        "group_code": str,
        "subgroup_code": str,
    },
)
gender_pairs = pd.read_csv(GENDER_DATA)
gender_pairs = gender_pairs.loc[gender_pairs["include_baseline"].astype(bool)].copy()
gender_pairs = gender_pairs.sort_values("pair_id").reset_index(drop=True)

assert df_analysis["occupation_code"].is_unique
assert len(gender_pairs) == 8
assert gender_pairs["male_term"].notna().all()
assert gender_pairs["female_term"].notna().all()

display(df_analysis.head())
display(gender_pairs)
display(df_analysis["submajor_group_code"].value_counts().sort_index().rename("n_occupation").to_frame())

## 3. Provenance dan pemuatan model

Metadata dihitung dari file lokal agar hasil dapat ditelusuri. Ukuran file, SHA-256, dimensi, jumlah vocabulary, versi Python, dan versi `fasttext-community` disimpan bersama output. Model tidak dipilih berdasarkan hasil SC-WEAT.

In [ ]:
def sha256_stream(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


loaded_models = {}
vocabulary_sets = {}
model_provenance_rows = []
fasttext_version = importlib.metadata.version("fasttext-community")

for spec in MODEL_SPECS.to_dict("records"):
    model = fasttext.load_model(str(spec["path"]))
    loaded_models[spec["model_id"]] = model
    vocabulary_sets[spec["model_id"]] = set(model.get_words())
    model_provenance_rows.append({
        "model_id": spec["model_id"],
        "model_name": spec["model_name"],
        "path": str(spec["path"]),
        "file_size_bytes": spec["path"].stat().st_size,
        "sha256": sha256_stream(spec["path"]),
        "dimension": model.get_dimension(),
        "vocabulary_size": len(vocabulary_sets[spec["model_id"]]),
        "corpus": spec["corpus"],
        "training_note": spec["training_note"],
        "source_url": spec["source_url"],
        "python_version": sys.version.split()[0],
        "fasttext_community_version": fasttext_version,
    })

model_provenance = pd.DataFrame(model_provenance_rows)
display(model_provenance)
assert set(loaded_models) == set(MODEL_SPECS["model_id"])
assert model_provenance["dimension"].eq(300).all()

## 4. Representasi istilah dan Jabatan

Tokenisasi dibuat sama untuk kedua model. Nama Jabatan multi-kata direpresentasikan sebagai rata-rata vektor token lalu dinormalisasi L2. Status vocabulary dibedakan dari validitas vektor karena fastText dapat membentuk subword vector untuk token yang tidak tercatat sebagai kata utuh.

In [ ]:
token_pattern = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*", flags=re.I)


def tokenize(text):
    return token_pattern.findall(str(text).lower())


def l2_normalize(vector):
    vector = np.asarray(vector, dtype=np.float32)
    norm = np.linalg.norm(vector)
    if norm == 0 or not np.isfinite(norm):
        return np.zeros_like(vector)
    return vector / norm


def phrase_vector(model, text):
    tokens = tokenize(text)
    vectors = [model.get_word_vector(token) for token in tokens]
    if not vectors:
        return np.zeros(model.get_dimension(), dtype=np.float32)
    return l2_normalize(np.mean(np.vstack(vectors), axis=0))


gender_terms = sorted(set(gender_pairs["male_term"]).union(gender_pairs["female_term"]))
term_vectors_by_model = {}
term_audit_rows = []
target_vectors_by_model = {}
target_audit_rows = []

for model_id, model in loaded_models.items():
    vocabulary = vocabulary_sets[model_id]
    term_vectors_by_model[model_id] = {}
    for term in gender_terms:
        vector = l2_normalize(model.get_word_vector(term))
        term_vectors_by_model[model_id][term] = vector
        term_audit_rows.append({
            "model_id": model_id,
            "term": term,
            "in_vocabulary": term in vocabulary,
            "vector_valid": bool(np.isfinite(vector).all() and np.linalg.norm(vector) > 0),
            "vector_norm": float(np.linalg.norm(vector)),
        })

    target_vectors_by_model[model_id] = {}
    for row in df_analysis.itertuples(index=False):
        vector = phrase_vector(model, row.occupation_text)
        target_vectors_by_model[model_id][row.occupation_code] = vector
        tokens = tokenize(row.occupation_text)
        target_audit_rows.append({
            "model_id": model_id,
            "occupation_code": row.occupation_code,
            "occupation_name_raw": row.occupation_name_raw,
            "token_count": len(tokens),
            "oov_token_count": sum(token not in vocabulary for token in tokens),
            "contains_oov": any(token not in vocabulary for token in tokens),
            "vector_valid": bool(np.isfinite(vector).all() and np.linalg.norm(vector) > 0),
        })

term_audit = pd.DataFrame(term_audit_rows)
target_audit = pd.DataFrame(target_audit_rows)
display(term_audit)
display(target_audit.groupby("model_id")["contains_oov"].agg(["count", "sum"]))

## 5. Perhitungan SC-WEAT dan association difference

Pipeline ini menggunakan adaptasi single-target: setiap Jabatan menjadi satu target, sedangkan `X` berisi istilah laki-laki dan `Y` berisi istilah perempuan. Tidak ada daftar karier–keluarga terpisah pada v3. Positif berarti asosiasi relatif Jabatan lebih dekat ke kelompok laki-laki; negatif berarti lebih dekat ke kelompok perempuan. Karena itu, penamaan dan kesimpulan dibatasi pada asosiasi relatif dalam embedding.

In [ ]:
def calculate_sc_weat(target_vector, male_vectors, female_vectors):
    male_similarity = male_vectors @ target_vector
    female_similarity = female_vectors @ target_vector
    association_difference = float(np.mean(male_similarity) - np.mean(female_similarity))
    all_similarity = np.concatenate([male_similarity, female_similarity])
    effect_size = association_difference / float(np.std(all_similarity, ddof=1))
    return {
        "association_difference": association_difference,
        "effect_size": effect_size,
        "male_similarity_mean": float(np.mean(male_similarity)),
        "female_similarity_mean": float(np.mean(female_similarity)),
    }


scweat_rows = []
for model_id in loaded_models:
    term_vectors = term_vectors_by_model[model_id]
    male_vectors = np.vstack([term_vectors[term] for term in gender_pairs["male_term"]])
    female_vectors = np.vstack([term_vectors[term] for term in gender_pairs["female_term"]])
    for row in df_analysis.itertuples(index=False):
        result = calculate_sc_weat(
            target_vectors_by_model[model_id][row.occupation_code],
            male_vectors,
            female_vectors,
        )
        scweat_rows.append({
            "model_id": model_id,
            "occupation_code": row.occupation_code,
            "occupation_name_raw": row.occupation_name_raw,
            "submajor_group_code": row.submajor_group_code,
            "submajor_group_name": row.submajor_group_name,
            "group_code": row.group_code,
            "group_name": row.group_name,
            **result,
        })

scweat_results = pd.DataFrame(scweat_rows)
scweat_results["direction"] = np.select(
    [scweat_results["effect_size"] > 0, scweat_results["effect_size"] < 0],
    ["positif", "negatif"],
    default="nol",
)
display(scweat_results.head())
display(scweat_results.groupby("model_id")["effect_size"].agg(["count", "mean", "median", "min", "max"]))

## 6. Exact permutation test dan koreksi FDR

Untuk setiap Jabatan dan model, seluruh pembagian delapan istilah menjadi kelompok laki-laki dan perempuan digunakan: `C(16, 8) = 12.870` partisi. P-value disimpan sebagai statistik inferensial sekunder dan dikoreksi dengan Benjamini–Hochberg pada seluruh pengujian Jabatan × model sebagai satu keluarga pengujian.

In [ ]:
combination_indices = np.array(list(itertools.combinations(range(16), 8)), dtype=np.int16)
assert len(combination_indices) == 12870

permutation_rows = []
for model_id in loaded_models:
    term_vectors = term_vectors_by_model[model_id]
    male_vectors = np.vstack([term_vectors[term] for term in gender_pairs["male_term"]])
    female_vectors = np.vstack([term_vectors[term] for term in gender_pairs["female_term"]])
    all_attribute_vectors = np.vstack([male_vectors, female_vectors])
    for row in df_analysis.itertuples(index=False):
        target_vector = target_vectors_by_model[model_id][row.occupation_code]
        similarities = all_attribute_vectors @ target_vector
        observed = float(np.mean(similarities[:8]) - np.mean(similarities[8:]))
        permutation_statistics = []
        for selected in combination_indices:
            selected_mask = np.zeros(16, dtype=bool)
            selected_mask[selected] = True
            permutation_statistics.append(
                float(np.mean(similarities[selected_mask]) - np.mean(similarities[~selected_mask]))
            )
        permutation_statistics = np.asarray(permutation_statistics)
        p_value = float((np.sum(np.abs(permutation_statistics) >= abs(observed)) + 1) / (len(permutation_statistics) + 1))
        permutation_rows.append({
            "model_id": model_id,
            "occupation_code": row.occupation_code,
            "observed_association_difference": observed,
            "permutation_p_value": p_value,
            "permutation_count": len(permutation_statistics),
        })

permutation_results = pd.DataFrame(permutation_rows)
scweat_results = scweat_results.merge(permutation_results, on=["model_id", "occupation_code"], how="left")
scweat_results["p_value_fdr_global"] = multipletests(
    scweat_results["permutation_p_value"], method="fdr_bh"
)[1]
scweat_results["significant_raw_0_05"] = scweat_results["permutation_p_value"] < 0.05
scweat_results["significant_fdr_0_05"] = scweat_results["p_value_fdr_global"] < 0.05
display(scweat_results.groupby("model_id")[["permutation_p_value", "p_value_fdr_global", "significant_raw_0_05", "significant_fdr_0_05"]].agg({"permutation_p_value": ["min", "median"], "p_value_fdr_global": ["min", "median"], "significant_raw_0_05": "sum", "significant_fdr_0_05": "sum"}))

## 7. Ringkasan Subgolongan Pokok

Ringkasan ini menunjukkan apakah pola model muncul secara agregat pada enam Subgolongan Pokok. Ringkasan tidak menggantikan analisis tingkat Jabatan dan tidak boleh dipakai untuk menghapus kelompok yang hasilnya tidak sesuai.

In [ ]:
group_summary = (
    scweat_results.groupby(["model_id", "submajor_group_code", "submajor_group_name"], as_index=False)
    .agg(
        n_occupation=("occupation_code", "count"),
        mean_effect_size=("effect_size", "mean"),
        median_effect_size=("effect_size", "median"),
        mean_association_difference=("association_difference", "mean"),
        negative_count=("direction", lambda values: int((values == "negatif").sum())),
        positive_count=("direction", lambda values: int((values == "positif").sum())),
        significant_raw_count=("significant_raw_0_05", "sum"),
        significant_fdr_count=("significant_fdr_0_05", "sum"),
    )
)
display(group_summary)

group_pivot = group_summary.pivot(index="submajor_group_code", columns="model_id", values="mean_effect_size")
display(group_pivot)
assert set(group_summary["model_id"]) == set(MODEL_SPECS["model_id"])
assert group_summary["n_occupation"].sum() == len(df_analysis) * len(MODEL_SPECS)

## 8. Perbandingan langsung antarmodel

Perbandingan utama memakai Jabatan yang sama pada kedua model. `model_difference` dihitung sebagai skor `wiki` dikurangi skor `cc_crawl`. Korelasi hanya menunjukkan kesamaan peringkat; korelasi tinggi tidak berarti besaran skor identik. Wilcoxon, paired sign-flip permutation, dan cluster bootstrap disediakan untuk menilai perbedaan antarmodel sebagai analisis komparatif, bukan untuk memilih model yang paling signifikan.

In [ ]:
model_ids = list(MODEL_SPECS["model_id"])
assert model_ids == ["cc_crawl", "wiki"]

comparison = scweat_results.pivot(
    index=["occupation_code", "occupation_name_raw", "submajor_group_code", "submajor_group_name", "group_code", "group_name"],
    columns="model_id",
    values=["association_difference", "effect_size", "permutation_p_value", "p_value_fdr_global", "direction"],
).reset_index()
comparison.columns = ["_".join(column).strip("_") if isinstance(column, tuple) else column for column in comparison.columns]
comparison["model_difference_wiki_minus_cc"] = comparison["effect_size_wiki"] - comparison["effect_size_cc_crawl"]
comparison["absolute_model_difference"] = comparison["model_difference_wiki_minus_cc"].abs()
comparison["same_direction"] = comparison["direction_wiki"].eq(comparison["direction_cc_crawl"])

model_comparison_rows = []
for model_id in model_ids:
    subset = scweat_results.loc[scweat_results["model_id"].eq(model_id)]
    model_comparison_rows.append({
        "model_id": model_id,
        "n_occupation": len(subset),
        "mean_effect_size": subset["effect_size"].mean(),
        "median_effect_size": subset["effect_size"].median(),
        "negative_count": int((subset["direction"] == "negatif").sum()),
        "positive_count": int((subset["direction"] == "positif").sum()),
        "minimum_p_value": subset["permutation_p_value"].min(),
        "raw_significant_count": int(subset["significant_raw_0_05"].sum()),
        "fdr_significant_count": int(subset["significant_fdr_0_05"].sum()),
    })
model_summary = pd.DataFrame(model_comparison_rows)
delta_values = comparison["model_difference_wiki_minus_cc"].to_numpy()
wilcoxon_result = wilcoxon(delta_values, alternative="two-sided", zero_method="wilcox")
rng = np.random.default_rng(RANDOM_SEED)
observed_delta_mean = float(np.mean(delta_values))
sign_flip_statistics = np.empty(10000, dtype=float)
for iteration in range(len(sign_flip_statistics)):
    signs = rng.choice(np.array([-1.0, 1.0]), size=len(delta_values))
    sign_flip_statistics[iteration] = float(np.mean(delta_values * signs))
paired_permutation_p = float((np.sum(np.abs(sign_flip_statistics) >= abs(observed_delta_mean)) + 1) / (len(sign_flip_statistics) + 1))

cluster_codes = comparison["group_code"].dropna().unique()
bootstrap_mean = np.empty(5000, dtype=float)
bootstrap_median = np.empty(5000, dtype=float)
for iteration in range(len(bootstrap_mean)):
    sampled_groups = rng.choice(cluster_codes, size=len(cluster_codes), replace=True)
    sampled = pd.concat([comparison.loc[comparison["group_code"].eq(group)] for group in sampled_groups], ignore_index=True)
    bootstrap_mean[iteration] = sampled["model_difference_wiki_minus_cc"].mean()
    bootstrap_median[iteration] = sampled["model_difference_wiki_minus_cc"].median()
model_difference_inference = pd.DataFrame([{
    "mean_delta_wiki_minus_cc": observed_delta_mean,
    "median_delta_wiki_minus_cc": float(np.median(delta_values)),
    "wilcoxon_statistic": float(wilcoxon_result.statistic),
    "wilcoxon_p_value": float(wilcoxon_result.pvalue),
    "paired_sign_flip_permutation_p_value": paired_permutation_p,
    "cluster_bootstrap_unit": "group_code tiga digit",
    "cluster_bootstrap_replicates": len(bootstrap_mean),
    "mean_delta_ci_2_5": float(np.percentile(bootstrap_mean, 2.5)),
    "mean_delta_ci_97_5": float(np.percentile(bootstrap_mean, 97.5)),
    "median_delta_ci_2_5": float(np.percentile(bootstrap_median, 2.5)),
    "median_delta_ci_97_5": float(np.percentile(bootstrap_median, 97.5)),
}])
spearman_value = spearmanr(comparison["effect_size_cc_crawl"], comparison["effect_size_wiki"]).statistic
kendall_value = kendalltau(comparison["effect_size_cc_crawl"], comparison["effect_size_wiki"]).statistic
model_agreement = pd.DataFrame([{
    "comparison": "cc_crawl_vs_wiki",
    "spearman_effect_size": float(spearman_value),
    "kendall_effect_size": float(kendall_value),
    "same_direction_rate": float(comparison["same_direction"].mean()),
    "mean_absolute_model_difference": float(comparison["absolute_model_difference"].mean()),
    "median_absolute_model_difference": float(comparison["absolute_model_difference"].median()),
    "sign_change_count": int((~comparison["same_direction"]).sum()),
}])
display(model_summary)
display(model_difference_inference)
display(model_agreement)
display(comparison.sort_values("absolute_model_difference", ascending=False).head(10))

## 9. Leave-one-pair-out pada masing-masing model

Analisis ini mengukur seberapa besar hasil berubah ketika satu pasangan lexical set dikeluarkan. Pasangan tidak dihapus permanen dan hasilnya tidak digunakan untuk memilih model.

In [ ]:
loo_rows = []
for model_id in loaded_models:
    for pair in gender_pairs.itertuples(index=False):
        keep_pairs = gender_pairs.loc[gender_pairs["pair_id"] != pair.pair_id]
        male_terms = keep_pairs["male_term"].tolist()
        female_terms = keep_pairs["female_term"].tolist()
        male_vectors = np.vstack([term_vectors_by_model[model_id][term] for term in male_terms])
        female_vectors = np.vstack([term_vectors_by_model[model_id][term] for term in female_terms])
        baseline = scweat_results.loc[scweat_results["model_id"].eq(model_id)].set_index("occupation_code")["effect_size"]
        for row in df_analysis.itertuples(index=False):
            loo_result = calculate_sc_weat(
                target_vectors_by_model[model_id][row.occupation_code], male_vectors, female_vectors
            )
            baseline_value = float(baseline.loc[row.occupation_code])
            loo_rows.append({
                "model_id": model_id,
                "occupation_code": row.occupation_code,
                "excluded_pair_id": pair.pair_id,
                "excluded_pair": f"{pair.male_term}-{pair.female_term}",
                "baseline_effect_size": baseline_value,
                "loo_effect_size": loo_result["effect_size"],
                "absolute_change": abs(loo_result["effect_size"] - baseline_value),
                "sign_changed": np.sign(loo_result["effect_size"]) != np.sign(baseline_value),
            })

loo_results = pd.DataFrame(loo_rows)
loo_summary = (
    loo_results.groupby(["model_id", "excluded_pair_id", "excluded_pair"], as_index=False)
    .agg(
        mean_absolute_change=("absolute_change", "mean"),
        median_absolute_change=("absolute_change", "median"),
        sign_change_count=("sign_changed", "sum"),
        n_occupation=("occupation_code", "count"),
    )
)
display(loo_summary.sort_values(["model_id", "mean_absolute_change"], ascending=[True, False]))

## 10. Visualisasi hasil perbandingan

Visualisasi ditampilkan langsung di notebook. Tidak ada gambar yang disimpan sebagai file.

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(9, 5))
sns.boxplot(data=scweat_results, x="model_id", y="effect_size", hue="model_id", legend=False)
plt.axhline(0, color="black", linewidth=1)
plt.title("Distribusi effect size SC-WEAT per model")
plt.xlabel("Model")
plt.ylabel("Effect size")
plt.show()

plt.figure(figsize=(7, 7))
sns.scatterplot(data=comparison, x="effect_size_cc_crawl", y="effect_size_wiki", hue="same_direction", palette={True: "#2a9d8f", False: "#e76f51"})
plt.axhline(0, color="black", linewidth=1)
plt.axvline(0, color="black", linewidth=1)
plt.title("Perbandingan effect size pada Jabatan yang sama")
plt.xlabel("cc.id.300.bin")
plt.ylabel("wiki.id.bin")
plt.show()

group_plot = group_summary.pivot(index="submajor_group_code", columns="model_id", values="mean_effect_size")
group_plot.plot(kind="bar", figsize=(10, 5), rot=0)
plt.axhline(0, color="black", linewidth=1)
plt.title("Mean effect size per Subgolongan Pokok")
plt.xlabel("Subgolongan Pokok")
plt.ylabel("Mean effect size")
plt.legend(title="Model")
plt.show()

## 11. Menyimpan output v3

Output menyimpan hasil tingkat Jabatan, permutation test, ringkasan kelompok, perbandingan antarmodel, leave-one-pair-out, dan provenance. File model tidak disalin ke repository.

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
scweat_results.to_csv(OUTPUT_DIR / "test_v3_hasil_scweat_per_model.csv", index=False, encoding="utf-8")
permutation_results.to_csv(OUTPUT_DIR / "test_v3_hasil_permutasi_per_model.csv", index=False, encoding="utf-8")
group_summary.to_csv(OUTPUT_DIR / "test_v3_ringkasan_subgolongan_per_model.csv", index=False, encoding="utf-8")
model_summary.to_csv(OUTPUT_DIR / "test_v3_ringkasan_model.csv", index=False, encoding="utf-8")
model_difference_inference.to_csv(OUTPUT_DIR / "test_v3_inferensi_perbedaan_model.csv", index=False, encoding="utf-8")
model_agreement.to_csv(OUTPUT_DIR / "test_v3_kesepakatan_model.csv", index=False, encoding="utf-8")
comparison.to_csv(OUTPUT_DIR / "test_v3_perbandingan_model_per_jabatan.csv", index=False, encoding="utf-8")
loo_results.to_csv(OUTPUT_DIR / "test_v3_sensitivitas_pasangan_per_model.csv", index=False, encoding="utf-8")
loo_summary.to_csv(OUTPUT_DIR / "test_v3_ringkasan_sensitivitas_pasangan_per_model.csv", index=False, encoding="utf-8")
model_provenance.to_csv(OUTPUT_DIR / "test_v3_audit_model.csv", index=False, encoding="utf-8")
term_audit.to_csv(OUTPUT_DIR / "test_v3_audit_istilah_gender.csv", index=False, encoding="utf-8")
target_audit.to_csv(OUTPUT_DIR / "test_v3_audit_target_oov.csv", index=False, encoding="utf-8")
display(pd.DataFrame({"output_file": sorted(path.name for path in OUTPUT_DIR.glob("test_v3_*.csv"))}))

## 12. Quality control final

QC memastikan kedua model memakai jumlah Jabatan, pasangan gender, dan jumlah partisi yang sama. QC tidak menyatakan bahwa model “benar” atau bahwa hasilnya signifikan.

In [ ]:
validation_rows = [
    {"check": "dua model dimuat", "status": len(loaded_models) == 2, "observed": ", ".join(sorted(loaded_models))},
    {"check": "jumlah pasangan gender", "status": len(gender_pairs) == 8, "observed": len(gender_pairs)},
    {"check": "jumlah partisi exact", "status": len(combination_indices) == 12870, "observed": len(combination_indices)},
    {"check": "jumlah Jabatan per model", "status": scweat_results.groupby("model_id").size().eq(len(df_analysis)).all(), "observed": scweat_results.groupby("model_id").size().to_dict()},
    {"check": "tidak ada effect size kosong", "status": scweat_results["effect_size"].notna().all(), "observed": int(scweat_results["effect_size"].isna().sum())},
    {"check": "perbandingan memiliki satu baris per Jabatan", "status": len(comparison) == len(df_analysis), "observed": len(comparison)},
]
validation = pd.DataFrame(validation_rows)
display(validation)
validation.to_csv(OUTPUT_DIR / "test_v3_validasi_pipeline.csv", index=False, encoding="utf-8")
assert validation["status"].all()
print("QC v3 lulus. Interpretasi substantif tetap harus menunggu pembacaan hasil kedua model.")